# Multi-Model Deep Learning Benchmarking & Hyperparameter Tuning
**Capstone Project in Data Science II (DS4105)**

This notebook provides **separate, independent code chunks** for every deep learning model evaluated in this comparative study:
1. **DistilBERT** (`distilbert-base-uncased`)
2. **DeBERTa-v3** (`microsoft/deberta-v3-small` / `base`)
3. **ELECTRA** (`google/electra-base-discriminator`)
4. **MobileBERT** (`google/mobilebert-uncased`)
5. **Longformer** (`allenai/longformer-base-4096`)
6. **PyTorch BiLSTM** (Bidirectional LSTM)
7. **Knowledge Distillation** (DeBERTa Teacher -> DistilBERT Student)

All models are evaluated on the exact same **70/15/15 Train/Validation/Test split** ($N = 6,061$ test reviews per model).

## Cell 1: Colab Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install HuggingFace dependencies
!pip install -q transformers datasets accelerate sentencepiece protobuf

## Cell 2: Configure Paths

In [ ]:
import os

folder_path = '/content/drive/MyDrive/DS4105_Capstone_Project/'
input_csv = os.path.join(folder_path, 'fake_reviews_cleaned.csv')
output_dir = os.path.join(folder_path, 'all_models_output')
trainer_script = os.path.join(folder_path, 'multi_model_trainer.py')

print(f"Input CSV: {input_csv}")
print(f"Output Dir: {output_dir}")

--- 
## Model Chunk 1: DistilBERT (`distilbert-base-uncased`)

In [ ]:
# Train DistilBERT
!python {trainer_script} \
    --input {input_csv} \
    --models distilbert \
    --epochs 3 \
    --batch_size 16 \
    --learning_rate 3e-5 \
    --warmup_ratio 0.1 \
    --weight_decay 0.01 \
    --output_dir {output_dir}

--- 
## Model Chunk 2: DeBERTa-v3 (`microsoft/deberta-v3-small`)
*Includes Warmup=0.1 & LR=2e-5 to prevent classification collapse.*

In [ ]:
# Train DeBERTa-v3 with Hyperparameter Tuning (LR=2e-5, Warmup=0.1, Weight Decay=0.01)
!python {trainer_script} \
    --input {input_csv} \
    --models deberta-v3 \
    --epochs 3 \
    --batch_size 16 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.1 \
    --weight_decay 0.01 \
    --output_dir {output_dir}

--- 
## Model Chunk 3: ELECTRA (`google/electra-base-discriminator`)

In [ ]:
# Train ELECTRA
!python {trainer_script} \
    --input {input_csv} \
    --models electra \
    --epochs 3 \
    --batch_size 16 \
    --learning_rate 3e-5 \
    --warmup_ratio 0.1 \
    --weight_decay 0.01 \
    --output_dir {output_dir}

--- 
## Model Chunk 4: MobileBERT (`google/mobilebert-uncased`)
*Includes Warmup=0.1 & LR=2e-5 to prevent classification collapse.*

In [ ]:
# Train MobileBERT with Hyperparameter Tuning (LR=2e-5, Warmup=0.1)
!python {trainer_script} \
    --input {input_csv} \
    --models mobilebert \
    --epochs 3 \
    --batch_size 16 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.1 \
    --weight_decay 0.01 \
    --output_dir {output_dir}

--- 
## Model Chunk 5: Longformer (`allenai/longformer-base-4096`)

In [ ]:
# Train Longformer
!python {trainer_script} \
    --input {input_csv} \
    --models longformer \
    --epochs 3 \
    --batch_size 16 \
    --learning_rate 3e-5 \
    --warmup_ratio 0.1 \
    --output_dir {output_dir}

--- 
## Model Chunk 6: PyTorch BiLSTM (Bidirectional LSTM)

In [ ]:
# Train PyTorch BiLSTM Classifier
!python {trainer_script} \
    --input {input_csv} \
    --models bilstm \
    --epochs 3 \
    --batch_size 16 \
    --output_dir {output_dir}

--- 
## Model Chunk 7: Knowledge Distillation (DeBERTa Teacher -> DistilBERT Student)

In [ ]:
# Train Knowledge Distillation (DeBERTa Teacher -> DistilBERT Student)
!python {trainer_script} \
    --input {input_csv} \
    --models distillation \
    --epochs 3 \
    --batch_size 16 \
    --output_dir {output_dir}

--- 
## Comparative Results Summary & Evaluation Table

In [ ]:
import pandas as pd

results_csv = os.path.join(output_dir, 'all_models_results.csv')
if os.path.exists(results_csv):
    df_results = pd.read_csv(results_csv)
    print("\n============================================================")
    print("  FINAL MULTI-MODEL EVALUATION TABLE (N = 6,061 Test Size)")
    print("============================================================")
    display(df_results.sort_values(by='f1_fake', ascending=False))
else:
    print(f"Results CSV not found yet at {results_csv}. Run one of the model chunks above!")